In [6]:
# ═══════════════════════════════════════════════════
# STEP 1 | Setup & Asset Preparation
#
#   Everything the workshop needs is prepared HERE, once.
#   After this cell, the pipeline runs even if the network drops.
#
#   Two things happen:
#     (1) Install pinned libraries
#     (2) Mount Drive, verify GPU, seal off the network,
#         and confirm every asset is in place
# ═══════════════════════════════════════════════════

# ── 1-1. Install (versions pinned for a reproducible workshop) ──
!pip install -q \
    langchain-core==1.4.8 \
    langchain-community==0.4.2 \
    langchain-huggingface==1.2.2 \
    langchain-text-splitters==1.1.2 \
    "sentence-transformers>=4,<5" \
    faiss-cpu==1.14.3 \
    pypdf==5.9.0 \
    "transformers>=4.51,<5"

#   langchain-core            Document objects, the LLM wrapper
#   langchain-community       FAISS wrapper
#   langchain-huggingface     encoder ↔ LangChain bridge
#   langchain-text-splitters  naive chunking (STEP 2, for final comparison)
#   sentence-transformers     encoder backend
#   faiss-cpu                 vector store
#   pypdf                     PDF → text (STEP 2)
#   transformers              Qwen3 (raised from 4.44 — Qwen3 needs >=4.51)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires reque

In [3]:
# ── 1-2. Mount Drive · verify GPU · seal the network · set paths ──
import os, torch
from google.colab import drive

# Seal off the Hugging Face Hub: everything is already local.
# This guarantees no cell will silently reach out to the internet.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# GPU is required for Qwen3. Stop now if it is missing,
# rather than failing deep inside STEP 10.
assert torch.cuda.is_available(), (
    "No GPU detected. Set: Runtime > Change runtime type > GPU (T4), then re-run."
)
print("GPU:", torch.cuda.get_device_name(0))

# Mount Drive
drive.mount("/content/drive")

# ── Paths: models only (everything else is defined where it is used) ──
BASE        = "/content/drive/MyDrive/DH2026_Workshop"
ENCODER_DIR = f"{BASE}/Encoder"
LLM_DIR     = f"{BASE}/LLM"

GPU: Tesla T4
Mounted at /content/drive


In [4]:
# ── 1-3. Confirm the models are in place (before we rely on them) ──
#   Only the pre-downloaded models are checked here.
#   The PDF is handled in STEP 2; the XML is produced in STEP 3.
checks = {
    "Encoder weights": f"{ENCODER_DIR}/model.safetensors",
    "Encoder pooling": f"{ENCODER_DIR}/1_Pooling/config.json",
    "LLM weights":     f"{LLM_DIR}/model.safetensors",
    "LLM config":      f"{LLM_DIR}/config.json",
}

missing = [name for name, path in checks.items() if not os.path.exists(path)]

for name, path in checks.items():
    mark = "✗" if name in missing else "✓"
    print(f"  {mark}  {name:16s} {path}")

assert not missing, f"Missing models: {missing}"
print("\nModels ready. The rest of the notebook runs offline.")

  ✓  Encoder weights  /content/drive/MyDrive/DH2026_Workshop/Encoder/model.safetensors
  ✓  Encoder pooling  /content/drive/MyDrive/DH2026_Workshop/Encoder/1_Pooling/config.json
  ✓  LLM weights      /content/drive/MyDrive/DH2026_Workshop/LLM/model.safetensors
  ✓  LLM config       /content/drive/MyDrive/DH2026_Workshop/LLM/config.json

Models ready. The rest of the notebook runs offline.


In [10]:
# ═══════════════════════════════════════════════════
# STEP 2 | PDF → text → naive chunks
#
#   These naive chunks are NOT used for retrieval.
#   They are set aside now and revealed only at the very end,
#   as the counter-example to the structured chunks (STEP 4) —
#   the "before" in a before/after argument about why structure matters.
#
#   Files are written to local disk (/content), not Drive:
#   they only need to live for this session.
# ═══════════════════════════════════════════════════

# ── 2-1. PDF → text ──
#   → pypdf
from pypdf import PdfReader

PDF_PATH = f"{BASE}/Source/The-Sustainable-Development-Goals-Report-2025.pdf"

reader   = PdfReader(PDF_PATH)
pages    = [(p.extract_text() or "") for p in reader.pages]
raw_text = "\n".join(pages)

with open("raw_text.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)

print(f"{len(pages)} pages, {len(raw_text):,} chars → raw_text.txt")

51 pages, 244,329 chars → raw_text.txt


In [11]:
# ── 2-2. text → naive chunks ──
#   → langchain-text-splitters
#   Blind 500-char cuts with 50-char overlap.
#   No structure, no headings — this is the naive baseline.
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter     = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
naive_chunks = splitter.split_text(raw_text)

with open("naive_chunks.txt", "w", encoding="utf-8") as f:
    for i, c in enumerate(naive_chunks):
        f.write(f"{'='*70}\n[chunk {i}]  {len(c)} chars\n{'='*70}\n{c}\n\n")

print(f"{len(naive_chunks)} naive chunks → naive_chunks.txt")

535 naive chunks → naive_chunks.txt


In [13]:
# ═══════════════════════════════════════════════════
# STEP 3 | Structure the PDF into hierarchical XML
#
#   A plain text dump loses the hierarchy (Goal > Subsection >
#   Paragraph) and mixes in noise (typesetting directives,
#   broken table fragments). build_xml.py rebuilds that hierarchy.
#
#   The XML produced here is written to LOCAL disk — you watch
#   the structuring happen. The next step reads the canonical
#   XML already prepared in Drive.
# ═══════════════════════════════════════════════════

BUILD_SCRIPT = f"{BASE}/Code/build_xml.py"

# build_xml.py takes:  [input PDF]  [output XML]
# PDF_PATH was defined in STEP 2. Output goes to local disk.
!python "{BUILD_SCRIPT}" "{PDF_PATH}" sdg_report_2025.xml

PDF → XML 변환 (Goal 1–17): /content/drive/MyDrive/DH2026_Workshop/Source/The-Sustainable-Development-Goals-Report-2025.pdf
Goal 페이지 범위: G1=10-11, G2=12-13, G3=14-17, G4=18-19, G5=20-21, G6=22-23, G7=24-25, G8=26-27, G9=28-29, G10=30-31, G11=32-33, G12=34-35, G13=36-37, G14=38-39, G15=40-41, G16=42-43, G17=44-45


  Goal  1 No poverty                               | kp= 4 sub= 6 para=16
  Goal  2 Zero hunger                              | kp= 4 sub= 9 para=18
  Goal  3 Good health and well-being               | kp= 4 sub=13 para=34
  Goal  4 Quality education                        | kp= 3 sub=10 para=27
  Goal  5 Gender equality                          | kp= 3 sub= 8 para=19
  Goal  6 Clean water and sanitation               | kp= 3 sub= 8 para=23
  Goal  7 Affordable and clean energy              | kp= 3 sub= 7 para=14
  Goal  8 Decent work and economic growth          | kp= 3 sub= 8 para=20
  Goal  9 Industry, innovation and infrastructure  | kp= 3 sub= 7 para=16
  Goal 10 Reduced in

In [14]:
# ═══════════════════════════════════════════════════
# STEP 4 | XML → chunks → LangChain Documents
#
#   One <subsection> = one chunk. The heading and body become
#   searchable text; the Goal number, name, pages, and heading
#   become metadata — the provenance that rides along with each hit.
#
#   Input is the CANONICAL XML in Drive (read-only), so every
#   participant starts from the exact same edition. A runtime
#   restart doesn't lose it — it lives in Drive, not local disk.
# ═══════════════════════════════════════════════════

# ── 4-1. XML → chunks ──
#   One <subsection> becomes one chunk.
import json, xml.etree.ElementTree as ET

XML_PATH = f"{BASE}/Source/sdg_report_2025.xml"   # canonical edition in Drive

root = ET.parse(XML_PATH).getroot()

chunks = []
for goal in root.findall("goal"):
    g_num, g_name, g_pages = int(goal.get("number")), goal.get("name"), goal.get("pages")

    for i, sub in enumerate(goal.findall("subsection")):
        heading = sub.get("heading")
        body    = "\n".join(p.text for p in sub.findall("paragraph") if p.text)

        chunks.append({
            "metadata": {                          # provenance. never embedded.
                "id":        f"G{g_num:02d}-S{i+1:02d}",
                "goal":      g_num,
                "goal_name": g_name,
                "pages":     g_pages,
                "heading":   heading,
            },
            "text": f"{heading}\n{body}",          # body. only this is embedded.
        })

with open("sdg_chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"{len(chunks)} chunks → sdg_chunks.jsonl")

143 chunks → sdg_chunks.jsonl


In [15]:
# ── 4-2. chunks → LangChain Documents ──
#   Same content, re-cast into the shape LangChain expects.
#     page_content → gets embedded (the search target)
#     metadata     → not embedded (used to cite provenance)
from langchain_core.documents import Document

docs = [Document(page_content=c["text"], metadata=c["metadata"]) for c in chunks]

print(docs[0].metadata)

{'id': 'G01-S01', 'goal': 1, 'goal_name': 'No poverty', 'pages': '10-11', 'heading': 'Revised poverty estimates show more people in extreme poverty, putting the 2030 goal further out of reach'}


In [7]:
# ═══════════════════════════════════════════════════
# STEP 5 | Load the encoder · embed every chunk
#
#   The encoder is loaded from Drive (local path), not the Hub —
#   the network is already sealed off (STEP 1). Same model as before,
#   just served from our own shelf.
#
#   Embeddings are regenerated live here. The vectors this encoder
#   produces in THIS environment become the canonical set — we do
#   not reuse any older embeddings file.
# ═══════════════════════════════════════════════════

# ── 5-1. Load the encoder (from Drive, local, on GPU) ──
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=ENCODER_DIR,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Encoder loaded from:", ENCODER_DIR)

Encoder loaded from: /content/drive/MyDrive/DH2026_Workshop/Encoder


In [8]:
# ── 5-2. Load the canonical chunks, then embed them ──
#   Input: the canonical chunks in Drive (read-only).
#   Each sentence becomes 384 numbers. On CPU this takes ~1–3 min. On GPU this takes ~1 second.
import json, time

CHUNKS_PATH = f"{BASE}/Source/sdg_chunks.jsonl"     # canonical chunks in Drive

chunks = [json.loads(line) for line in open(CHUNKS_PATH, encoding="utf-8")]

t0 = time.time()
vectors = embeddings.embed_documents([c["text"] for c in chunks])
print(f"{len(vectors)} embeddings done · {time.time()-t0:.1f}s\n")

print(f"dimensions: {len(vectors[0])}")
print(f"first chunk, first 10 values: {[round(x, 4) for x in vectors[0][:10]]}\n")

with open("sdg_embeddings.jsonl", "w", encoding="utf-8") as f:
    for c, v in zip(chunks, vectors):
        f.write(json.dumps({**c, "vector": v}, ensure_ascii=False) + "\n")

print("→ sdg_embeddings.jsonl")

143 embeddings done · 1.1s

dimensions: 384
first chunk, first 10 values: [-0.0741, -0.0232, 0.021, 0.069, 0.0258, 0.0122, -0.118, 0.0536, -0.0602, 0.0348]

→ sdg_embeddings.jsonl


In [12]:
# ═══════════════════════════════════════════════════
# STEP 6 | Build the vector store (FAISS)
#
#   → faiss-cpu, langchain-community
#   The vectors from STEP 5 go straight in — nothing is recomputed.
#   Vectors and source text live separately; an index number joins them.
# ═══════════════════════════════════════════════════
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.vectorstores import FAISS

store = FAISS.from_embeddings(
    text_embeddings=list(zip([c["text"] for c in chunks], vectors)),
    embedding=embeddings,
    metadatas=[c["metadata"] for c in chunks],
    ids=[c["metadata"]["id"] for c in chunks],
)

print(f"{store.index.ntotal} × {store.index.d} dims · {type(store.index).__name__}\n")

i = 0
print("vector store :", store.index.reconstruct(i)[:4].round(4), "...")
print("id table     :", i, "→", store.index_to_docstore_id[i])
print("text store   :", store.docstore.search(store.index_to_docstore_id[i]).metadata["heading"][:50])

143 × 384 dims · IndexFlatL2

vector store : [-0.0741 -0.0232  0.021   0.069 ] ...
id table     : 0 → G01-S01
text store   : Revised poverty estimates show more people in extr


In [9]:
# ═══════════════════════════════════════════════════
# STEP 7 | A question becomes a vector too
#
#   The same encoder that embedded the chunks now embeds the query.
#   The only difference: a query carries no metadata.
# ═══════════════════════════════════════════════════

# ── 7-1. Embed a query ──
query = "How many people still live in extreme poverty?"

qv = embeddings.embed_query(query)

print(f"query : {query}")
print(f"vector: {len(qv)} dims\n")

print("  dim       value")
print("  ────────────────")
for i in range(10):
    print(f"  {i:3d}   {qv[i]:+.4f}")
print("   ⋮        ⋮")
for i in range(374, 384):
    print(f"  {i:3d}   {qv[i]:+.4f}")

query : How many people still live in extreme poverty?
vector: 384 dims

  dim       value
  ────────────────
    0   +0.0672
    1   -0.0524
    2   -0.0036
    3   +0.0941
    4   -0.0382
    5   -0.0046
    6   -0.0592
    7   +0.0583
    8   -0.0708
    9   +0.0126
   ⋮        ⋮
  374   +0.0087
  375   -0.0921
  376   +0.0398
  377   +0.0101
  378   -0.0512
  379   -0.0629
  380   +0.0663
  381   -0.0827
  382   -0.0374
  383   +0.0296


In [10]:
# ── 7-2. Try it yourself — what does my question become? ──
#   Re-run this cell as often as you like. It affects nothing downstream.
import numpy as np

my_query = input("Enter your question (English): ")
mv = np.array(embeddings.embed_query(my_query))

print(f"\n{my_query}")
print(f"→ {len(mv)}-dim vector\n")

for r in range(0, 384, 8):
    print(f"  [{r:3d}] " + "  ".join(f"{x:+.4f}" for x in mv[r:r+8]))

Enter your question (English): How about the North Korea`s conditions?

How about the North Korea`s conditions?
→ 384-dim vector

  [  0] -0.0335  +0.0831  +0.0369  -0.0105  +0.0312  -0.0235  -0.0608  -0.0589
  [  8] -0.1421  +0.0160  +0.0318  -0.0011  +0.0495  +0.0369  +0.0288  -0.0448
  [ 16] -0.0111  -0.0453  -0.0068  +0.0132  -0.0210  +0.0068  +0.0494  +0.0185
  [ 24] -0.1338  -0.0669  +0.1031  -0.0168  +0.0340  -0.0324  -0.0136  +0.0110
  [ 32] -0.0356  +0.0498  +0.0694  +0.0269  -0.0338  +0.0283  -0.0502  -0.0392
  [ 40] +0.0273  -0.0807  +0.0708  -0.0599  +0.0596  +0.0221  -0.0176  -0.0635
  [ 48] -0.0412  -0.0138  +0.0932  -0.0117  -0.0652  -0.0019  +0.0706  -0.0404
  [ 56] -0.0822  -0.0782  +0.0058  -0.0058  +0.0452  -0.0150  -0.0775  -0.0044
  [ 64] +0.0906  +0.0368  +0.0498  +0.1452  -0.0410  +0.1213  +0.0136  +0.0281
  [ 72] -0.0088  +0.0361  -0.0664  -0.0224  +0.0197  -0.0189  +0.0446  -0.0200
  [ 80] +0.0360  +0.0062  -0.0025  -0.0370  +0.0045  +0.0059  -0.0736  -0.0672
 

In [18]:
# ═══════════════════════════════════════════════════
# STEP 8 | Distance between the query vector and the index
#
#   Search is nothing more than measuring distances and sorting.
#   Here we do it by hand — no FAISS — to see what FAISS does for us.
# ═══════════════════════════════════════════════════
import numpy as np

query = "How many people still live in extreme poverty?"
qv    = embeddings.embed_query(query)                  # the query as a vector

V = store.index.reconstruct_n(0, store.index.ntotal)   # all 143 chunk vectors
Q = np.array(qv)                                        # the one query vector

print(f"chunk vectors : {len(V)}, each {len(V[0])} numbers")
print(f"query vector  : 1, {len(Q)} numbers\n")

# ── measure the distance to each chunk ──
#    (square the differences, sum all 384, take the square root)
distances = []
for v in V:
    diff = v - Q                        # 384 differences
    dist = np.sqrt((diff ** 2).sum())   # square → sum → square root
    distances.append(dist)

print(f"measured {len(distances)} distances, one per chunk.\n")

# ── sort nearest first ──
order = np.argsort(distances)

hand_top3 = [store.index_to_docstore_id[int(i)] for i in order[:3]]

for rank, i in enumerate(order[:3], 1):
    print(f"#{rank}  distance {distances[i]:.4f}   {store.index_to_docstore_id[int(i)]}")

chunk vectors : 143, each 384 numbers
query vector  : 1, 384 numbers

measured 143 distances, one per chunk.

#1  distance 0.8608   G01-S01
#2  distance 0.8651   G01-S02
#3  distance 0.9866   G11-S01


In [17]:
# ═══════════════════════════════════════════════════
# STEP 9 | FAISS does the same — then attach provenance
#
#   What we computed by hand in STEP 8, FAISS does in one line.
#   We confirm the results match, then attach the source
#   (which Goal, which subsection) to every hit.
# ═══════════════════════════════════════════════════

# ── 9-1. Ask FAISS, compare with the hand-computed result ──
query = "How many people still live in extreme poverty?"

results    = store.similarity_search_with_score(query, k=3)
faiss_top3 = [doc.metadata["id"] for doc, _ in results]

print(f"query: {query}\n")
print("STEP 8 (by hand) :", hand_top3)
print("STEP 9 (FAISS)   :", faiss_top3)
print("match            :", hand_top3 == faiss_top3)

query: How many people still live in extreme poverty?

STEP 8 (by hand) : ['G01-S01', 'G01-S02', 'G11-S01']
STEP 9 (FAISS)   : ['G01-S01', 'G01-S02', 'G11-S01']
match            : True


In [16]:
# ── 9-2. Attach provenance to search results ──
#   Metadata is never searched — it rides along with the hit.
def search_with_sources(q, k=3):
    for rank, (doc, dist) in enumerate(store.similarity_search_with_score(q, k=k), 1):
        m = doc.metadata
        print(f"[{rank}]  distance {dist:.4f}")
        print(f"     {m['id']} · Goal {m['goal']} {m['goal_name']} · pp. {m['pages']}")
        print(f"     {m['heading']}")
        print()
        print(doc.page_content)
        print("─" * 70)

query = "How many people still live in extreme poverty?"
search_with_sources(query)

[1]  distance 0.7410
     G01-S01 · Goal 1 No poverty · pp. 10-11
     Revised poverty estimates show more people in extreme poverty, putting the 2030 goal further out of reach

Revised poverty estimates show more people in extreme poverty, putting the 2030 goal further out of reach
The World Bank revised global poverty estimates using updated price data and national poverty lines from over 160 countries in June 2025. The international poverty line was raised from $2.15 (2017 purchasing power parity (PPP)) to $3.00 (2021 PPP). Under the new threshold, 1.5 billion people escaped poverty between 1990 and 2022 – compared to 1.3 billion under the previous line. However, the update leads to an upward revision of extreme poverty.  In 2025, an estimated 808 million people will be living in extreme poverty – up from the previous estimate of 677 million – representing 9.9 per cent of the world’s population, or 1 in 10 people.
Eradicating extreme poverty by 2030 appears highly unlikely due to sl

In [28]:
# ═══════════════════════════════════════════════════
# STEP 10 | Generation with provenance (Qwen3-0.6B)
#
#   The retriever finds evidence; the LLM turns it into an answer.
#   Every answer is shown together with the sources it rests on.
#
#   Qwen3 differs from Flan-T5 in three ways:
#     ① causal, not seq2seq → AutoModelForCausalLM
#     ② a chat model → wrap input in the chat template
#     ③ it echoes the prompt → we slice the answer back out
# ═══════════════════════════════════════════════════

# ── 10-1. Load Qwen3, wrap it as a LangChain LLM ──
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

hf_logging.set_verbosity_error()          # hide generation-flag notices

tokenizer = AutoTokenizer.from_pretrained(LLM_DIR)          # local, defined in STEP 1
model = AutoModelForCausalLM.from_pretrained(
    LLM_DIR,
    dtype="auto",
    device_map="cuda",
)
print("Loaded:", model.config.model_type,
      f"· {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")


class Qwen3LLM(LLM):
    """Qwen3 behind LangChain's LLM interface: call it with llm.invoke(prompt)."""
    max_new_tokens: int = 512

    @property
    def _llm_type(self) -> str:
        return "qwen3-local"

    def _call(self, prompt: str, stop: Optional[List[str]] = None,
              run_manager: Any = None, **kwargs) -> str:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,          # no <think> block, answer only
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        prompt_len = inputs.input_ids.shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,            # greedy: same answer every time
                pad_token_id=tokenizer.eos_token_id,
            )
        answer_ids = out[0][prompt_len:]    # drop the echoed prompt
        return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()


llm = Qwen3LLM()
print("Ready.")

Loaded: qwen3 · 596M params
Ready.


In [24]:
# ── 10-2. BEFORE: ask without any context ──
#   No retrieval, no evidence. The model answers from memory alone —
#   possibly outdated, possibly wrong, with no source to check.
import textwrap

query = "How many people still live in extreme poverty?"

answer = llm.invoke(query)
print(textwrap.fill(answer, width=80))

As of the latest data (2023), the number of people living in extreme poverty is
approximately **1.5 billion**. This figure includes individuals who are unable
to meet the basic needs of food, shelter, and healthcare. The situation has been
growing in recent years due to factors such as climate change, economic
inequality, and global crises.


In [27]:
# ── 10-3. AFTER: answer grounded in retrieved sources ──
#   Retrieve evidence → feed it to the LLM → answer WITH its sources.
#   Now the answer rests on the report, and every claim is traceable.
import textwrap

def ask_with_sources(q, k=3):
    # 1. retrieve
    results = store.similarity_search_with_score(q, k=k)

    # 2. build the context from retrieved chunks
    context = "\n\n".join(doc.page_content for doc, _ in results)
    prompt = (
        "Use only the following sources to answer the question. "
        "If the answer is not in the sources, say you cannot find it.\n\n"
        f"{context}\n\n"
        f"Question: {q}"
    )

    # 3. generate
    answer = llm.invoke(prompt)

    # 4. show the answer, then the sources it rests on
    print(textwrap.fill(answer, width=80))
    print("\nSources")
    for doc, _ in results:
        m = doc.metadata
        print(f"  [{m['id']}] Goal {m['goal']} {m['goal_name']} · pp. {m['pages']}")

query = "How many people still live in extreme poverty?"
ask_with_sources(query)

The number of people still living in extreme poverty is 808 million as stated in
the text.

Sources
  [G01-S01] Goal 1 No poverty · pp. 10-11
  [G01-S02] Goal 1 No poverty · pp. 10-11
  [G11-S01] Goal 11 Sustainable cities and communities · pp. 32-33
